# Car Price Prediction — Milestone 2: Data Preparation (v3 — Final)

Google Colab → Files → **Upload to session storage**: `car_price_dataset.csv`

[Kaggle Dataset](https://www.kaggle.com/datasets/deepcontractor/car-price-prediction-challenge)

---
**Corrections applied in v3:**
- Fixed Pipeline DataFrame reconstruction (was truncating to 13 cols instead of 34)
- Replaced all `fillna(0)` patches with proper upfront imputation
- Added `.describe()` summary of new features before analysis plots
- Added KDE age-group plot to notebook (was only in PDF before)
- Tightened `add_features()` to not rely on global `raw_eng` variable

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style='ticks', palette='Set2')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 4)})
PAL = sns.color_palette('Set2')

print('Setup complete. Seed:', RANDOM_SEED)

## 2. Load & Inspect Raw Data

In [ ]:
df_raw = pd.read_csv('car_price_dataset.csv')
print('Shape:', df_raw.shape)
df_raw.head(10)

In [ ]:
print('Dtypes:\n', df_raw.dtypes)
print('\nSample values for object columns:')
for col in df_raw.select_dtypes('object').columns:
    print(f'  {col}: {df_raw[col].unique()[:5]}')

## 3. Data Cleaning

### 3.1 Hidden Missing Value Analysis

This dataset encodes missing values as `"-"` instead of `NaN` — particularly in the `Levy` (vehicle tax) column.

In [ ]:
dash_counts = (df_raw == '-').sum()
dash_counts = dash_counts[dash_counts > 0]
dash_pct    = (dash_counts / len(df_raw) * 100).round(2)

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.bar(dash_pct.index, dash_pct.values, color=PAL[0], width=0.5)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_ylabel('Missing / Placeholder (%)')
ax.set_title('Hidden Missing Values (Dash Placeholders) per Column')
ax.set_ylim(0, dash_pct.max() * 1.4)
sns.despine(); plt.tight_layout(); plt.show()
print(dash_pct)

### 3.2 Type Cleaning & Imputation

In [ ]:
df = df_raw.copy()

# Step 1 — Replace dash placeholders with NaN
df.replace('-', np.nan, inplace=True)

# Step 2 — Strip units and cast to numeric
df['Mileage'] = pd.to_numeric(
    df['Mileage'].astype(str).str.replace(' km', '', regex=False)
                             .str.replace(',', '', regex=False), errors='coerce')

df['Engine volume'] = pd.to_numeric(
    df['Engine volume'].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')

df['Levy'] = pd.to_numeric(
    df['Levy'].astype(str).str.replace(',', '', regex=False), errors='coerce')

df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Step 3 — Drop rows where target Price is missing (cannot be imputed)
before = len(df)
df.dropna(subset=['Price'], inplace=True)
print(f'Rows dropped — missing Price: {before - len(df)}')

# Step 4 — Impute Levy with median (preferred over mean: Levy is right-skewed)
levy_median = df['Levy'].median()
df['Levy'].fillna(levy_median, inplace=True)
print(f'Levy imputed with median: {levy_median:.0f}')

# Step 5 — Impute all remaining numeric NaNs with column median
for col in df.select_dtypes(include=np.number).columns:
    n = df[col].isnull().sum()
    if n > 0:
        df[col].fillna(df[col].median(), inplace=True)
        print(f'  {col}: {n} values imputed')

# Step 6 — Drop rows with any missing categorical value
cat_cols = df.select_dtypes('object').columns
before = len(df)
df.dropna(subset=cat_cols, inplace=True)
print(f'Rows dropped — missing categoricals: {before - len(df)}')
print(f'\nClean dataset shape: {df.shape}')
print(f'Remaining NaNs: {df.isnull().sum().sum()}')

## 4. Outlier Investigation

Both ends of the price distribution are inspected before applying any filter.

In [ ]:
df_before = df.copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['Price'].clip(upper=200000), bins=80,
             color='#5B8DB8', edgecolor='none', alpha=0.85)
axes[0].set_title('Price Distribution (Linear Scale)')
axes[0].set_xlabel('Price ($)'); axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].hist(np.log1p(df['Price']), bins=70, color=PAL[2], edgecolor='none', alpha=0.85)
axes[1].set_title('log(1 + Price) — Shows Full Range')
axes[1].set_xlabel('log(1 + Price)'); axes[1].set_ylabel('Count')
plt.suptitle('Price Distribution Before Outlier Filtering', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

print('Lowest 10 prices:',  df['Price'].nsmallest(10).values)
print('Highest 10 prices:', df['Price'].nlargest(10).values)

In [ ]:
# Lower bound only — prices below $500 are data entry errors (e.g. $1, $2)
# Upper bound: NOT applied — luxury/exotic cars at $100k+ are valid real-world listings.
# Removing them would bias the model to underpredict for the premium segment.

n_before = len(df)
df = df[df['Price'] >= 500].copy()
print(f'Removed {n_before - len(df)} rows with Price < $500')
print(f'Remaining: {len(df):,} rows')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(np.log1p(df_before['Price']), bins=60, color=PAL[3], edgecolor='none', alpha=0.85)
axes[0].set_title('Before Filtering'); axes[0].set_xlabel('log(1+Price)'); axes[0].set_ylabel('Count')
axes[1].hist(np.log1p(df['Price']), bins=60, color=PAL[4], edgecolor='none', alpha=0.85)
axes[1].set_title('After Removing Price < $500'); axes[1].set_xlabel('log(1+Price)'); axes[1].set_ylabel('Count')
plt.suptitle('Effect of Outlier Filtering on Price Distribution', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 5. Distributions of Key Numeric Features

In [ ]:
num_feats = ['Prod. year', 'Mileage', 'Engine volume', 'Levy', 'Airbags', 'Doors']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, feat, c in zip(axes.flatten(), num_feats, PAL):
    data = df[feat].clip(upper=df[feat].quantile(0.99))
    ax.hist(data, bins=35, color=c, edgecolor='none', alpha=0.85)
    ax.set_title(feat); ax.set_xlabel(feat); ax.set_ylabel('Count')
plt.suptitle('Distributions of Key Numeric Features', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Full correlation heatmap of original numeric features
num_df = df.select_dtypes(include=np.number)
num_df = num_df[[c for c in num_df.columns if c != 'Price'] + ['Price']]
corr   = num_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, linewidths=0.4, ax=ax,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.75},
            annot_kws={'size': 8})
ax.set_title('Correlation Matrix of Numeric Features', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout(); plt.show()

print('Top correlations with Price:')
print(corr['Price'].drop('Price').sort_values().to_string())

## 6. Train / Test Split

The split is performed **before** any feature engineering or scaling to prevent data leakage.
Stratified by **price decile bins** so both sets reflect the same price composition across the full range.

In [ ]:
df['_bin'] = pd.qcut(df['Price'], q=10, labels=False, duplicates='drop')
X = df.drop(columns=['Price', '_bin'])
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=df['_bin'])

print(f'Training samples : {len(X_train):,}  ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test samples     : {len(X_test):,}   ({len(X_test)/len(X)*100:.0f}%)')

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(np.log1p(y.min()), np.log1p(y.max()), 50)
ax.hist(np.log1p(y_train), bins=bins, alpha=0.65,
        label=f'Train (n={len(y_train):,})', color=PAL[0])
ax.hist(np.log1p(y_test),  bins=bins, alpha=0.65,
        label=f'Test  (n={len(y_test):,})',  color=PAL[3])
ax.set_xlabel('log(1 + Price)'); ax.set_ylabel('Count')
ax.set_title('Target Distribution: Train vs Test — Stratified 80/20 Split')
ax.legend(); sns.despine(); plt.tight_layout(); plt.show()

print('\nTrain price statistics:'); print(y_train.describe().round(0))
print('\nTest price statistics:');  print(y_test.describe().round(0))

## 7. Feature Engineering

Four new hand-crafted features are derived from the existing columns using domain knowledge about used car valuation.
All features are computed after the split and applied consistently to both train and test sets.

| Feature | Formula | Reasoning |
|---|---|---|
| `Car_Age` | 2024 − Prod. year | Captures depreciation timeline directly |
| `Km_Per_Year` | Mileage ÷ (Car_Age + 1) | Normalises usage intensity by ownership duration |
| `Has_Turbo` | 1 if "Turbo" in original engine string | Turbocharged engines command a significant price premium |
| `Airbag_Tier` | Binned: 0→T0, 1-4→T1, 5-8→T2, 9+→T3 | Safety tier — reduces effect of outlier airbag counts |

In [ ]:
# Store the raw engine string before it was cleaned — needed to extract Turbo flag
# This is passed in explicitly so add_features() has no hidden dependency on globals
raw_engine_strings = df_raw['Engine volume'].astype(str)

def add_features(X_in, raw_eng_series):
    """
    Add domain-informed features to a feature DataFrame.
    raw_eng_series: the original (pre-cleaning) Engine volume column as strings.
    """
    o = X_in.copy()

    # Feature 1: Vehicle Age
    o['Car_Age'] = (2024 - o['Prod. year'].astype(int)).clip(lower=0)

    # Feature 2: Annual Kilometre Rate
    # +1 in denominator prevents division by zero for brand-new cars (Car_Age = 0)
    o['Km_Per_Year'] = o['Mileage'] / (o['Car_Age'] + 1)

    # Feature 3: Turbo flag — extracted from raw string before unit-stripping removed it
    o['Has_Turbo'] = (raw_eng_series.loc[X_in.index]
                      .str.lower()
                      .str.contains('turbo')
                      .astype(int))

    # Feature 4: Airbag safety tier
    o['Airbag_Tier'] = pd.cut(
        o['Airbags'], bins=[-1, 0, 4, 8, 100], labels=[0, 1, 2, 3]
    ).astype(int)

    return o

X_train = add_features(X_train, raw_engine_strings)
X_test  = add_features(X_test,  raw_engine_strings)

new_feats = ['Car_Age', 'Km_Per_Year', 'Has_Turbo', 'Airbag_Tier']
print('New features created:', new_feats)
print()
print('Descriptive statistics of new features (training set):')
print(X_train[new_feats].describe().round(2))

## 8. Analysis of New Features

Each new feature is analysed with the same methods used in Milestone 1: distribution histograms, relationship to Price, and correlation analysis.

In [ ]:
# Distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat, c in zip(axes.flatten(), new_feats, PAL):
    data = X_train[feat].clip(upper=X_train[feat].quantile(0.98))
    ax.hist(data, bins=30, color=c, edgecolor='none', alpha=0.85)
    ax.set_title(f'Distribution of {feat}')
    ax.set_xlabel(feat); ax.set_ylabel('Count')
plt.suptitle('Distributions of New Hand-Crafted Features (Training Set)', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Regression plots — new continuous features vs Price (with trend line + confidence band)
sample = X_train.copy()
sample['Price'] = y_train.values
sample = sample.sample(min(2500, len(sample)), random_state=RANDOM_SEED)

reg_feats = [('Car_Age', 'Car Age (years)'),
             ('Km_Per_Year', 'Km Per Year'),
             ('Engine volume', 'Engine Volume (L)')]
colors_r  = ['#2196F3', '#4CAF50', '#FF5722']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (feat, label), col in zip(axes, reg_feats, colors_r):
    x  = sample[feat].values
    yy = sample['Price'].values
    valid = np.isfinite(x) & np.isfinite(yy)
    slope, intercept, r, p, _ = stats.linregress(x[valid], yy[valid])
    xline = np.linspace(x[valid].min(), x[valid].max(), 100)
    yline = slope * xline + intercept
    ax.scatter(x, yy, alpha=0.15, s=8, color=col)
    ax.plot(xline, yline, color='red', linewidth=2, label=f'r = {r:.2f}')
    ax.fill_between(xline,
                    yline - yy[valid].std() * 0.35,
                    yline + yy[valid].std() * 0.35,
                    alpha=0.15, color='red')
    ax.set_xlabel(label); ax.set_ylabel('Price ($)')
    ax.set_title(f'{label} vs Price')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(fontsize=9)
plt.suptitle('Key Features vs Car Price — Regression Plots', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Violin plots — categorical new features vs Price
tp = pd.DataFrame({
    'Has_Turbo':   X_train['Has_Turbo'].map({0: 'Non-Turbo', 1: 'Turbo'}),
    'Airbag_Tier': X_train['Airbag_Tier'],
    'Price':       y_train.values
})
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.violinplot(data=tp, x='Has_Turbo',   y='Price', palette='Set2',
               ax=axes[0], inner='quartile', cut=0)
axes[0].set_title('Price by Engine Type (Turbo vs Non-Turbo)')
axes[0].set_xlabel('Engine Type'); axes[0].set_ylabel('Price ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

sns.violinplot(data=tp, x='Airbag_Tier', y='Price', palette='Set2',
               ax=axes[1], inner='quartile', cut=0)
axes[1].set_title('Price by Airbag Safety Tier (0=none → 3=high)')
axes[1].set_xlabel('Airbag Tier'); axes[1].set_ylabel('Price ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.suptitle('New Categorical Features vs Price', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# KDE — Price density split by car age group
tp2 = pd.DataFrame({'Car_Age': X_train['Car_Age'], 'Price': y_train.values})
terciles = pd.qcut(tp2['Car_Age'], q=3,
                   labels=['Young (0–6 yr)', 'Mid (7–13 yr)', 'Old (14+ yr)'])
tp2['Age Group'] = terciles

fig, ax = plt.subplots(figsize=(9, 4))
for group, color in zip(['Young (0–6 yr)', 'Mid (7–13 yr)', 'Old (14+ yr)'], PAL):
    subset = tp2[tp2['Age Group'] == group]['Price']
    sns.kdeplot(np.log1p(subset), ax=ax, label=group,
                fill=True, alpha=0.30, color=color)
ax.set_xlabel('log(1 + Price)'); ax.set_ylabel('Density')
ax.set_title('Price Density by Car Age Group — KDE on log-price')
ax.legend(title='Age Group'); sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap of new features + Price
new_corr = pd.DataFrame({f: X_train[f] for f in new_feats})
new_corr['Price'] = y_train.values

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(new_corr.corr(), annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.85})
ax.set_title('Correlation Matrix of New Features + Price')
plt.tight_layout(); plt.show()

## 9. Encoding Categorical Variables

### Why not Ordinal Encoding for high-cardinality features?
Ordinal encoding assigns integers such as Toyota=0, BMW=1, Ford=2. This implies a numeric ordering (Toyota < BMW < Ford) that does not exist.
The model would treat the difference between two manufacturers as a meaningful distance, which is incorrect.

### Target Encoding (high-cardinality: Manufacturer, Model, Color)
Each category is replaced with the **mean Price of that category in the training set**.
BMW maps to ~$45,000 because that is what BMWs actually sell for on average — a genuinely informative signal.
The mapping is learned on the training set only and applied to the test set, so there is no leakage.

### One-Hot Encoding (low-cardinality: Fuel type, Gear box type, Drive wheels, etc.)
Creates binary indicator columns. No ordering is implied between categories.

In [ ]:
cat_cols  = X_train.select_dtypes('object').columns.tolist()
card      = {c: X_train[c].nunique() for c in cat_cols}
high_card = [c for c, n in card.items() if n > 8]
low_card  = [c for c, n in card.items() if n <= 8]

print('Target Encoding  (high cardinality):')
for c in high_card: print(f'  {c:25s} — {card[c]} unique values')
print('\nOne-Hot Encoding (low cardinality):')
for c in low_card:  print(f'  {c:25s} — {card[c]} unique values')

In [ ]:
# ── Target Encoding ───────────────────────────────────────────────────────────
# Fit the mean-price mapping on training set ONLY, then apply to test
global_mean  = y_train.mean()
target_maps  = {}

for col in high_card:
    target_maps[col] = y_train.groupby(X_train[col].astype(str)).mean().to_dict()
    X_train[col] = X_train[col].astype(str).map(target_maps[col]).fillna(global_mean)
    X_test[col]  = X_test[col].astype(str).map(target_maps[col]).fillna(global_mean)

# Visualise: show mean price per Manufacturer as proof
mfr_means = pd.Series(target_maps['Manufacturer']).sort_values(ascending=False)
fig, ax   = plt.subplots(figsize=(9, 4))
ax.bar(mfr_means.index, mfr_means.values, color=PAL[0])
ax.set_xlabel('Manufacturer'); ax.set_ylabel('Mean Price ($)')
ax.set_title('Target Encoding — Mean Price per Manufacturer\n(this value replaces the category string)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right'); sns.despine(); plt.tight_layout(); plt.show()

print('Manufacturer encoding sample (first 5):')
print(dict(list(target_maps['Manufacturer'].items())[:5]))

In [ ]:
# ── One-Hot Encoding ──────────────────────────────────────────────────────────
X_train = pd.get_dummies(X_train, columns=low_card, drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=low_card, drop_first=True)

# Align: test may be missing a dummy column if a rare category only appears in train
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f'Feature count after encoding: {X_train.shape[1]}')
print(f'All dtypes numeric: {X_train.select_dtypes(exclude=np.number).shape[1] == 0}')
print(f'Remaining NaNs: {X_train.isnull().sum().sum()}')

## 10. Scaling via sklearn Pipeline + ColumnTransformer

### Why a Pipeline?
Manual scaling (`scaler.fit_transform(X_train)` then `scaler.transform(X_test)`) is correct but error-prone — it is easy to accidentally apply `fit_transform` to the test set.
A `Pipeline` makes the correct order **structural**: `fit_transform()` on train, `transform()` on test, with no possibility of mixing them up. The pipeline can also be extended in Milestone 3 by appending a model step.

### Why MinMaxScaler?
Maps all features to [0, 1] while **preserving the shape of each distribution** exactly. Chosen over StandardScaler because our features include heavily skewed distributions (Mileage, Levy) where the standard deviation is not a meaningful normalisation unit.

In [ ]:
# All columns are now numeric after encoding
# Convert any remaining bool columns to int (sklearn requires numeric)
bool_cols = X_train.select_dtypes(include='bool').columns.tolist()
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols]  = X_test[bool_cols].astype(int)

all_feature_cols = X_train.columns.tolist()
print(f'NaNs before pipeline: {X_train.isnull().sum().sum()}')
print(f'(These come from target-encoded categories with very few training examples)')

# ── Build Pipeline ─────────────────────────────────────────────────────────────
# The pipeline has two sequential steps per column:
#   1. SimpleImputer(median)  — handles any residual NaNs from target encoding
#   2. MinMaxScaler           — normalises to [0, 1]
# This is the correct professional approach: all preprocessing is inside the pipeline,
# so fit_transform() on train and transform() on test are structurally leak-free.

from sklearn.impute import SimpleImputer

preprocessor = ColumnTransformer(
    transformers=[
        ('impute_scale', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  MinMaxScaler())
        ]), all_feature_cols)
    ],
    remainder='drop'  # explicit: no column left unhandled
)

preprocessing_pipeline = Pipeline(steps=[('preprocessing', preprocessor)])

# Fit on training data ONLY, then transform both sets
X_train_arr = preprocessing_pipeline.fit_transform(X_train)
X_test_arr  = preprocessing_pipeline.transform(X_test)

# Reconstruct as DataFrames with correct column names
X_train_sc = pd.DataFrame(X_train_arr, columns=all_feature_cols, index=X_train.index)
X_test_sc  = pd.DataFrame(X_test_arr,  columns=all_feature_cols, index=X_test.index)

print(f'Pipeline fitted successfully.')
print(f'X_train_sc shape : {X_train_sc.shape}')
print(f'X_test_sc  shape : {X_test_sc.shape}')
print(f'Remaining NaNs   : {X_train_sc.isnull().sum().sum()}')  # must be 0

In [ ]:
# Visualise scaling effect on 3 key features
demo = ['Mileage', 'Car_Age', 'Levy']
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for i, feat in enumerate(demo):
    axes[0, i].hist(X_train[feat], bins=35, color=PAL[i], edgecolor='none', alpha=0.85)
    axes[0, i].set_title(f'{feat} — Original Scale')
    axes[0, i].set_xlabel(feat); axes[0, i].set_ylabel('Count')

    axes[1, i].hist(X_train_sc[feat], bins=35, color=PAL[i+3], edgecolor='none', alpha=0.85)
    axes[1, i].set_title(f'{feat} — After MinMaxScaler [0, 1]')
    axes[1, i].set_xlabel(f'{feat} (scaled)'); axes[1, i].set_ylabel('Count')
plt.suptitle('MinMaxScaler via Pipeline — Shape Preserved, Range Normalised to [0, 1]',
             fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 11. Feature Selection — Permutation Importance with Real Threshold

A `GradientBoostingRegressor` is trained on the scaled training data and **Permutation Importance** is computed.
This technique measures how much the model's prediction error increases when a feature's values are randomly shuffled.
A large increase means the model relied heavily on that feature; a near-zero value indicates noise.

**Threshold: importance > 0.002**
This is a data-driven cutoff that genuinely reduces dimensionality. A threshold of `> 0` (used previously) is meaningless because it keeps every feature regardless of quality.

In [ ]:
gbm = GradientBoostingRegressor(
    n_estimators=100, max_depth=4,
    learning_rate=0.1, random_state=RANDOM_SEED
)
gbm.fit(X_train_sc, y_train)

perm = permutation_importance(
    gbm, X_train_sc, y_train,
    n_repeats=10, random_state=RANDOM_SEED, n_jobs=-1
)
perm_s = pd.Series(perm.importances_mean,
                   index=X_train_sc.columns).sort_values(ascending=False)

print(f'Total features before selection: {len(perm_s)}')
print('\nAll importance scores:')
print(perm_s.round(5).to_string())

In [ ]:
IMPORTANCE_THRESHOLD = 0.002

selected = perm_s[perm_s >  IMPORTANCE_THRESHOLD].index.tolist()
dropped  = perm_s[perm_s <= IMPORTANCE_THRESHOLD].index.tolist()

print(f'Threshold        : {IMPORTANCE_THRESHOLD}')
print(f'Features KEPT    : {len(selected)}')
print(f'Features DROPPED : {len(dropped)}')
print(f'\nSelected features: {selected}')
print(f'Dropped features : {dropped}')

X_train_final = X_train_sc[selected].copy()
X_test_final  = X_test_sc[selected].copy()
print(f'\nFinal shape — Train: {X_train_final.shape}, Test: {X_test_final.shape}')

In [ ]:
# Feature importance chart with threshold line (green = selected, grey = dropped)
fig, ax = plt.subplots(figsize=(10, max(5, len(perm_s) * 0.35)))
bar_colors = ['#3D9970' if v > IMPORTANCE_THRESHOLD else '#cccccc'
              for v in perm_s.sort_values().values]
perm_s.sort_values().plot.barh(ax=ax, color=bar_colors)
ax.axvline(IMPORTANCE_THRESHOLD, color='red', linestyle='--', linewidth=2,
           label=f'Threshold = {IMPORTANCE_THRESHOLD}')
ax.set_xlabel('Mean Permutation Importance (MSE increase when shuffled)')
ax.set_title(f'Feature Selection — {len(selected)} kept (green), {len(dropped)} dropped (grey)')
ax.legend(); sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Final correlation heatmap of selected features + Price
plot_feats = selected[:12] if len(selected) > 12 else selected
corr_f = X_train_final[plot_feats].copy()
corr_f['Price'] = y_train.values

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_f.corr(), dtype=bool))
sns.heatmap(corr_f.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.4, ax=ax,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
            annot_kws={'size': 8})
ax.set_title('Correlation Matrix — Selected Features + Price',
             fontsize=12, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

## 12. Export Prepared Data

In [ ]:
X_train_final.to_csv('X_train_prepared.csv', index=False)
X_test_final.to_csv('X_test_prepared.csv',  index=False)
y_train.to_csv('y_train_prepared.csv',       index=False)
y_test.to_csv('y_test_prepared.csv',         index=False)

print('Files saved successfully:')
print(f'  X_train_prepared.csv  → {X_train_final.shape}')
print(f'  X_test_prepared.csv   → {X_test_final.shape}')
print(f'  y_train_prepared.csv  → {y_train.shape}')
print(f'  y_test_prepared.csv   → {y_test.shape}')